# Pupper, zero-shot: type a request, watch one general policy play it

Type what you'd say to Pupper. Gemini writes the motion as keyframes, a checker makes sure it's
feasible (balance, joint limits, physics), and **one general tracking policy plays it in simulation**.
That policy was trained once, on ~1,000 generated motions, and never on the one you just asked for.

You get a side-by-side video: **Gemini's reference | plain open-loop replay | the general tracker**.

**To run:** Runtime → Change runtime type → a **GPU**. Run the first cell (about 3 minutes, once), then
the second cell as often as you like (about a minute per request). Nothing here touches a robot.

In [ ]:
# @title 1. Setup (about 3 minutes, once)
import os, sys, subprocess
os.environ["MUJOCO_GL"] = "egl"  # off-screen rendering for the videos (before anything imports mujoco)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU: Runtime -> Change runtime type -> GPU")
REPO, BRANCH = "https://github.com/MlsBli/gemini_pupper", "final-project-kflib"
PUPPER_MJLAB, COMMIT = "https://github.com/cs123-stanford/pupper-mjlab", "aa475e9a393a209b29fd5de7e0ee10f3de643230"
if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q {PUPPER_MJLAB} /content/pupper-mjlab
  !git -C /content/pupper-mjlab checkout -q {COMMIT}
  !pip install -q -e /content/pupper-mjlab mediapy
if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} --depth 1 {REPO} /content/gemini_pupper
sys.path[:0] = ["/content/pupper-mjlab/src", "/content/gemini_pupper/final_project"]
os.chdir("/content/gemini_pupper/final_project")

from IPython.display import Video, display
from kflib import demo
from kflib.llm import LLM
CHECKPOINT = "/content/gemini_pupper/final_project/demo/pgt_v2.pt"  # the general tracker (v2)
assert os.path.exists(CHECKPOINT), f"the tracker's weights are missing ({CHECKPOINT}): pull the latest {BRANCH}"
llm = LLM("colab")  # Colab's built-in Gemini: no API key needed
print("ready | keyframes by", llm.model, "| tracker", os.path.basename(CHECKPOINT))
RUN = 0

## 2. Ask Pupper for something

Anything a small robot dog could do in place in a few seconds, starting and ending standing.
Numbers are welcome ("lift your back right paw 4 cm"). Some ideas: *"do a happy wiggle, then bow"*,
*"look left and right like you heard a noise"*, *"crouch down slowly and shake your head no"*.

In [ ]:
# @title Make it and play it (about a minute)
REQUEST = "Shake your body like a wet dog"  # @param {type:"string"}
RUN += 1
result = demo.run(REQUEST, llm, CHECKPOINT, f"/content/demo_runs/{RUN:02d}")
print(demo.describe(result))
if result.get("video"):
  display(Video(result["video"], embed=True, width=1080))

### Optional: look around in 3D

The same tracker playing the last motion, in an interactive viewer (opens a public link; the
translucent ghost is the reference). Stop the cell to close it.

In [ ]:
# @title 3D viewer for the last request
!python -m mjlab.tasks.tracking.config.pupper.play_motion {result['npz']} --checkpoint {CHECKPOINT} --viewer viser --share True --loops 3